> **Interactive Jupyter notebook implementing the interpolation, Iwasawa power series approximation, Newton slope extraction, and ramification classification pipeline. Includes commented test runs and verification examples.**  

In [1]:
def power_series(D, p, i, verb=0):
    r"""
    Computes the Iwasawa power series associated to a quadratic Dirichlet character, 
    the Iwasawa power series approximation, Newton slope extraction, and ramification classification.
        
    INPUT:
    - ``D`` -- Fundamental discriminant.
    - ``p`` -- Prime number p (odd). 
    - ``i`` -- Teichmüller twist i (0 <= i < p - 1).
    - ``verb`` -- verbosity: only characters with lambda >= verb are printed 
    
    OUTPUT: 
    - ``fp`` -- Approximating polynomial of the Iwasawa power series in Qp[x].
    - ``lambda_inv`` -- Lambda-invariant of the power series.
    - ``v0`` -- Valuation of the constant coefficient of the power series.

    AUTHOR: Heiko Knospe, 2026
    """
    
    K = Qp(p, prec=50)
    v = K.valuation()
    R = PolynomialRing(K, 'x')

    if not Integer(D).is_fundamental_discriminant():
        print("Not a fundamental discriminant")
        return None, None, None

    if (D>0 and i%2==1) or (D<0 and i%2==0):
        print("Trivial p-adic L-function")
        return None, None, None

    cond = abs(D)
        
    if gcd(cond, p) > 1:
        print("Not supported. gcd(|D|, p) > 1.")
        return None, None, None

    if i not in range(0, p - 1):
        print("Teichmuller twist invalid")
        return None, None, None

    chip = kronecker(D, p)
    
    if i == 1 and chip == 1:
        print(f"p={p}, twist={i}, D={D} ==> Trivial zero case")
        return None, None, None
      
    # Compute special values of the p-adic L-function
    # Variable: T = (1 + cond * p)^(-j + 1) - 1
    points = [
        (K((1 + cond * p)**(-j + 1) - 1), pari.lfun(D, 1 - j) * (1 - chip * p**(j - 1))) 
        for j in range(p - 1 + i, 15 * p, p - 1)
    ]

    # Interpolate to obtain an approximating power series
    fp = R.lagrange_polynomial(points)
    v0 = v(fp[0])  # Valuation of constant coefficient
        
    # Compute lambda-invariant
    # Search for lambda up to 15
    # Large lambda values are very rare
    lambda_bound = 15
    j = 0
    while j <= lambda_bound:
        if v(fp[j]) <= 0: 
            break
        j += 1
    
    if j > lambda_bound:
        print(f'p={p}, twist={i}, Fundamental Discriminant={D} ==> lambda>={j}, v(const)={v0}')
        return fp, lambda_bound + 1, v0 
            
    slopes = []
    discstring = ''
    ramtype = [', unramified field extension', ', ramified field extension', ', trivial field extension']

    if j > 0:
        fR = fp.truncate(j + 1) 
        try:       
            slopes = fR.newton_slopes()[0:j]  # Slopes of distinguished polynomial
        except Exception:
            slopes = 'NA'
        
        # Further analyze slopes with multiplicity >= 2
        # (Multiplicity 1 corresponds to a linear factor of the distinguished polynomial, hence a root in Zp)
        if j >= 2 and isinstance(slopes, list):
            myslopes = fR.newton_slopes(repetition=False)
            for myslope in myslopes:
                # Case 1: Multiplicity 2 (Quadratic factor)
                if slopes.count(myslope) == 2:
                    try:
                        g = fp.factor_of_slope(myslope)
                        Disc = g.discriminant()
                        vD = v(Disc)
                        if vD % 2 == 1:
                            rt = ramtype[1]  # Ramified quadratic extension
                        else:
                            # Unit part projected to GF(p)
                            u0 = Disc.unit_part().residue()  
                            if u0.is_square():
                                rt = ramtype[2]  # Splits in Qp (two linear roots)
                            else:
                                rt = ramtype[0]  # Unramified quadratic extension
                        
                        discstring += ', slope=' + str(myslope) + ' v(Disc)=' + str(vD) + rt
                    except Exception:
                        discstring += ', slope=' + str(myslope) + ' ***Error***'
                        
                # Case 2: Multiplicity > 2 (Degree >= 3 factors, rare)
                if slopes.count(myslope) > 2:  
                    try:
                        g = fp.factor_of_slope(myslope)
                        Disc = g.discriminant()
                        vD = v(Disc)                      
                        discstring += ', slope=' + str(myslope) + ' v(Disc)=' + str(vD)

                        # Fractional slopes correspond strictly to ramified extensions
                        if myslope not in ZZ:
                            discstring +=  ramtype[1] + ' of degree ' + str(slopes.count(myslope))
                        else:
                            s = ZZ(myslope)
                            for myfactor, multiplicity in g.factor():
                                for _ in range(multiplicity):
                                    d = myfactor.degree()
                                    if d == 1:
                                        discstring += ', zero in Qp'
                                    else:
                                        is_unram = False
                                        is_ram = False

                                        X = myfactor.parent().gen()
                                        h = sum(myfactor[k] * (p**(k * s - d * s)) * X**k for k in range(d + 1))
                                        
                                        # Try Sage extension first
                                        try:
                                            E = K.extension(h, names='alpha')
                                            e_idx = E.ramification_index()
                                            if e_idx == 1:
                                                is_unram = True
                                            elif e_idx > 1:
                                                is_ram = True
                                        except Exception:
                                            # Fallback algebraic invariants
                                            v_fac_disc = v(myfactor.discriminant())
                                            v_norm = v_fac_disc - s * d * (d - 1)

                                            if v_fac_disc % 2 == 1:
                                                is_ram = True  # Odd valuation is strictly ramified
                                            elif d == 2 and v_fac_disc % 2 == 0:
                                                is_unram = True  # Even valuation for d=2 is strictly unramified
                                            elif v_norm == 0:
                                                is_unram = True  # v_norm == 0 is strictly unramified
                                            else:
                                                try:
                                                    hbar = h.change_ring(GF(p))
                                                    if hbar.is_irreducible():
                                                        is_unram = True
                                                except Exception:
                                                    pass

                                        if is_unram:
                                            discstring += ramtype[0] + ' of degree ' + str(d)
                                        elif is_ram:
                                            discstring += ramtype[1] + ' of degree ' + str(d)
                    except Exception:
                        discstring += ', slope=' + str(myslope) + ' ***Error***'
    if j >= verb: # print line if lambda >= verb
        print(f'p={p}, twist={i}, Fundamental Discriminant={D} ==> lambda={j}, v(const)={v0}, slopes={slopes}{discstring}')   
    return fp, j, v0

In [2]:
p=3 # prime
i=1 # Teichmuller twist
lamb=[0]*15
vals=[0]*15

for D in range(-2,-1000,-1):
    if Integer(D).is_fundamental_discriminant()==False or gcd(D,p)!=1:        
        continue
    if i==1 and kronecker(D,p)==1:
        continue
    fp, j, v0 = power_series(D,p,i,1) # print out characters with lambda>=1
    lamb[j]+=1
    vals[v0]+=1

print(f'\np={p}, twist={i}, \ndistribution-lambda=      {lamb},\ndistribution-v(constant)= {vals}' )        
     

p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-139 ==> lambda=1, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-199 ==> lambda=1, v(const)=2, slopes=[2]
p=3, twist=1, Fundamental Discriminant=-211 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2 v(Disc)=1, ramified field extension
p=3, twist=1, Fundamental Discriminant=-244 ==> lambda=1, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-247 ==> lambda=1, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-283 ==> lambda=1, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-307 ==> lambda=1, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-331 ==> lambda=1, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-367 ==> lambda=1, v(const)=2, slopes=[2]
p=3, twist=1, Fundamental Discriminant=-379 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2 v(Disc)=1, ramified field extension
p=

In [3]:
D=456
p=5
i=0
fp, j, v0 = power_series(D,p,i)
print(fp) # approximate power series, long output
F=Qp(p,prec=7,print_mode='digits')
R.<T> = PolynomialRing(F)
print(R(fp)) # short output

p=5, twist=0, Fundamental Discriminant=456 ==> lambda=3, v(const)=1, slopes=[1/3, 1/3, 1/3], slope=1/3 v(Disc)=2, ramified field extension of degree 3
(4 + 3*5^3 + 2*5^4 + 5^6 + 2*5^7 + 5^8 + 2*5^10 + 3*5^11 + 5^12 + 2*5^13 + 2*5^14 + 4*5^15 + 5^16 + 4*5^17 + 2*5^18 + 5^20 + 2*5^21 + 3*5^24 + 3*5^25 + 3*5^27 + 2*5^28 + 5^30 + O(5^31))*x^17 + (4 + 4*5^2 + 5^3 + 2*5^4 + 3*5^5 + 3*5^6 + 2*5^7 + 5^8 + 4*5^9 + 2*5^10 + 3*5^11 + 3*5^12 + 3*5^13 + 5^14 + 2*5^15 + 2*5^16 + 5^18 + 5^19 + 3*5^20 + 3*5^22 + 4*5^23 + 4*5^24 + 5^25 + 4*5^26 + 2*5^27 + 2*5^28 + 5^29 + 3*5^30 + 5^31 + O(5^32))*x^16 + (2 + 3*5 + 5^2 + 5^4 + 4*5^5 + 3*5^6 + 4*5^7 + 4*5^9 + 3*5^10 + 2*5^11 + 4*5^12 + 3*5^13 + 2*5^14 + 3*5^15 + 3*5^16 + 5^18 + 4*5^20 + 4*5^21 + 4*5^22 + 3*5^23 + 5^24 + 3*5^25 + 4*5^26 + 3*5^28 + 4*5^29 + 2*5^30 + 5^31 + O(5^33))*x^15 + (4 + 5 + 2*5^2 + 5^3 + 4*5^4 + 3*5^5 + 2*5^6 + 5^7 + 4*5^8 + 2*5^10 + 2*5^11 + 3*5^12 + 5^14 + 3*5^15 + 3*5^16 + 3*5^17 + 5^18 + 3*5^21 + 2*5^22 + 2*5^24 + 2*5^25 + 3*5^29

In [4]:
D=2635213
p=3
i=0
fp, j, v0 = power_series(D,p,i)

p=3, twist=0, Fundamental Discriminant=2635213 ==> lambda=7, v(const)=1, slopes=[1/7, 1/7, 1/7, 1/7, 1/7, 1/7, 1/7], slope=1/7 v(Disc)=6, ramified field extension of degree 7


In [5]:
D=456
p=5
i=0
fp, j, v0 = power_series(D,p,i)

p=5, twist=0, Fundamental Discriminant=456 ==> lambda=3, v(const)=1, slopes=[1/3, 1/3, 1/3], slope=1/3 v(Disc)=2, ramified field extension of degree 3


In [6]:
# Verification against some examples in Ernvall-Metsänkylä, Computation of the zeros of p-adic L-functions, Math. of Comp. Vol. 58, 1992. 
# ->ok. Note: reverse order of p-adic digits in E-V
EV=[[-1399,3,1],[21,11,4],[2536,3,0],[2732,3,0],[-184,5,3],[2504,5,2],[456,5,0],[-2283,5,3],[24,7,0]]
for D,p,i in EV:
    F=Qp(p,prec=7,print_mode='digits')
    R.<T> = PolynomialRing(F)
    fp, j, v0 = power_series(D,p,i)
    print('F(T)=',R(fp).truncate(6)) # .map_coefficients(lambda c: c.add_bigoh(5)))

p=3, twist=1, Fundamental Discriminant=-1399 ==> lambda=2, v(const)=3, slopes=[2, 1]
F(T)= ...01121220*T^5 + ...2221001*T^4 + ...2120102*T^3 + ...1210112*T^2 + ...21001120*T + ...0000002000
p=11, twist=4, Fundamental Discriminant=21 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2 v(Disc)=1, ramified field extension
F(T)= ...293A697*T^5 + ...6A39661*T^4 + ...399A2A30*T^3 + ...61A8591*T^2 + ...2182A520*T + ...66041960
p=3, twist=0, Fundamental Discriminant=2536 ==> lambda=2, v(const)=2, slopes=[1, 1], slope=1 v(Disc)=2, trivial field extension
F(T)= ...1211022*T^5 + ...02211220*T^4 + ...100202200*T^3 + ...2120012*T^2 + ...101121200*T + ...000001100
p=3, twist=0, Fundamental Discriminant=2732 ==> lambda=2, v(const)=2, slopes=[1, 1], slope=1 v(Disc)=3, ramified field extension
F(T)= ...1120221*T^5 + ...1000122*T^4 + ...1211122*T^3 + ...2112021*T^2 + ...21012220*T + ...000001100
p=5, twist=3, Fundamental Discriminant=-184 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2 v(Disc)

In [7]:
# Examples in Chapter 6 of our paper
Examp=[[3989,3,0],[77,3,0],[2908,3,0],[-709255,3,1]]
for D,p,i in Examp:
    F=Qp(p,prec=7,print_mode='digits')
    R.<T> = PolynomialRing(F)
    fp, j, v0 = power_series(D,p,i)
    print('F(T)=',R(fp).truncate(6)) # .map_coefficients(lambda c: c.add_bigoh(5)))

p=3, twist=0, Fundamental Discriminant=3989 ==> lambda=2, v(const)=3, slopes=[2, 1]
F(T)= ...1200002*T^5 + ...102220100*T^4 + ...1010101*T^3 + ...2202211*T^2 + ...10220010*T + ...0000002000
p=3, twist=0, Fundamental Discriminant=77 ==> lambda=2, v(const)=1, slopes=[1/2, 1/2], slope=1/2 v(Disc)=1, ramified field extension
F(T)= ...2211102*T^5 + ...1211102*T^4 + ...0121202*T^3 + ...2012112*T^2 + ...12010010*T + ...00000110
p=3, twist=0, Fundamental Discriminant=2908 ==> lambda=2, v(const)=2, slopes=[1, 1], slope=1 v(Disc)=2, unramified field extension
F(T)= ...0000121000*T^5 + ...1021202*T^4 + ...2021011*T^3 + ...2011001*T^2 + ...22020210000*T + ...000001100
p=3, twist=1, Fundamental Discriminant=-709255 ==> lambda=4, v(const)=4, slopes=[1, 1, 1, 1], slope=1 v(Disc)=12, unramified field extension of degree 4
F(T)= ...1121012*T^5 + ...1021022*T^4 + ...21010110*T^3 + ...0101212000000*T^2 + ...2000122000*T + ...00000220000
